In [1]:
import re
import unicodedata

# ======================================================
# Função para normalização
# ======================================================

def normalizar(texto):
    texto = texto.lower().strip()

    # Remove acentos
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    # Remove espaços duplicados
    texto = re.sub(r'\s+', ' ', texto)

    return texto


In [2]:


# ======================================================
# Leitura do dicionário
# ======================================================

dic_words = []

with open("PrejudicePT-br.dic", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        if line.startswith("%"):
            continue

        # ignora linhas das categorias
        if re.match(r'^\d+\s', line):
            continue

        palavra = line.split()[0]

        palavra = normalizar(palavra)

        dic_words.append(palavra)

print("Entradas do dicionário:", len(dic_words))




Entradas do dicionário: 842


In [3]:
# ======================================================
# Leitura do TXT
# ======================================================

with open("valores.txt", encoding="utf-8") as f:
    texto = f.read()

texto = normalizar(texto)

# Remove comentários entre parênteses
texto = re.sub(r'\(.*?\)', '', texto)

categorias = [
    "capacitismo",
    "gordofobia",
    "intolerancia religiosa",
    "lgbtqia+fobia",
    "lgbtfobia",
    "misoginia",
    "xenofobia",
    "racismo",
    "etarismo",
    "preconceito politico"
]

for categoria in categorias:
    texto = texto.replace(normalizar(categoria), "")

tokens = re.split(r'[,;\n]+', texto)

txt_words = set()

for token in tokens:

    token = normalizar(token)

    if token != "":
        txt_words.add(token)

print("Palavras únicas do TXT:", len(txt_words))


# ======================================================
# Comparação considerando o *
# ======================================================

encontradas = []
nao_encontradas = []

for entrada in dic_words:

    if "*" in entrada:

        regex = "^" + re.escape(entrada).replace("\\*", ".*") + "$"

        achou = any(re.match(regex, palavra) for palavra in txt_words)

    else:

        achou = entrada in txt_words

    if achou:
        encontradas.append(entrada)
    else:
        nao_encontradas.append(entrada)


# ======================================================
# Resultados
# ======================================================

print("\n==============================")
print("RESULTADOS")
print("==============================")

print(f"Entradas no dicionário : {len(dic_words)}")
print(f"Palavras únicas no TXT : {len(txt_words)}")
print(f"Entradas encontradas   : {len(encontradas)}")
print(f"Entradas não encontradas: {len(nao_encontradas)}")

percentual = 100 * len(encontradas) / len(dic_words)

print(f"\nCobertura do dicionário: {percentual:.2f}%")


# ======================================================
# Salva listas
# ======================================================

with open("palavras_encontradas.txt", "w", encoding="utf-8") as f:

    for palavra in sorted(encontradas):
        f.write(palavra + "\n")

with open("palavras_nao_encontradas.txt", "w", encoding="utf-8") as f:

    for palavra in sorted(nao_encontradas):
        f.write(palavra + "\n")

print("\nArquivos gerados:")
print(" - palavras_encontradas.txt")
print(" - palavras_nao_encontradas.txt")

Palavras únicas do TXT: 362

RESULTADOS
Entradas no dicionário : 842
Palavras únicas no TXT : 362
Entradas encontradas   : 137
Entradas não encontradas: 705

Cobertura do dicionário: 16.27%

Arquivos gerados:
 - palavras_encontradas.txt
 - palavras_nao_encontradas.txt
